# segment-line-intersect-2d — ex2: batched segments vs a single infinite line

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `segment-line-intersect-2d`. Running the final beacon cell reports progress against the `Geometry: Segment-line intersect 2-D` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Segment-line intersect 2-D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`segment-line-intersect-2d`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "segment-line-intersect-2d"
DD_SUBTOPIC = "Geometry: Segment-line intersect 2-D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 2-D segment vs line intersection — deepening refresher

A single intersection is a 2×2 linear system. For a BATCH of `N` segments vs ONE infinite line, broadcast the construction and solve all `N` systems in a single `t.linalg.solve` call:

```python
d = S1 - S0                          # (N, 2)  segment directions
e = L1 - L0                          # (2,)    one line direction
A = t.stack([d, -e.expand_as(d)], dim=-1)   # (N, 2, 2): columns d and -e
b = L0 - S0                          # (N, 2)
ts = t.linalg.solve(A, b)            # (N, 2)  — [t_i, s_i] per segment
```

**Hit mask.** `hit = (ts[..., 0] >= 0) & (ts[..., 0] <= 1)` — closed interval on segment parameter, line parameter unconstrained.

**Parallel-segment handling.** When a segment is parallel to the line, the corresponding 2×2 is singular and `linalg.solve` raises. For a batched call, the entire batch raises even if just one segment is parallel. The robust workaround is to mask via determinant before solving (covered separately in `singular-matrix-mask-trick`).

### Exercise 2 — batched segments vs a single infinite line

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `t.linalg.solve` to a batched `(N, 2, 2)` system to intersect `N` segments with one infinite line in 2-D, returning the per-segment t-parameter and a boolean hit mask.
> Keywords: 2D-geometry, batched-linalg-solve, broadcast, hit-mask
> ```

**KCs targeted:** `batched-2x2-linear-system`, `broadcast-line-direction`

Implement `ex2_batched_seg_line(S0, S1, L0, L1)`. Find the intersections of `N` segments with ONE infinite line in 2-D.

1. `S0`, `S1` are `(N, 2)` float tensors — N segment endpoints. Segment direction: `d = S1 - S0` (shape `(N, 2)`).
2. `L0`, `L1` are `(2,)` float tensors — two points defining the infinite line. Line direction: `e = L1 - L0` (shape `(2,)`).
3. Build the per-segment 2×2 system `A @ [t, s].T = (L0 - S0)`:
   ```python
   A = t.stack([d, -e.expand_as(d)], dim=-1)   # (N, 2, 2)
   b = L0 - S0                                  # (N, 2)
   ts = t.linalg.solve(A, b)                    # (N, 2)
   ```
   The second column of `A` is `-e` broadcast over `N`.
4. Hit mask: `hit = (ts[..., 0] >= 0) & (ts[..., 0] <= 1)` — closed segment interval, line unconstrained. Shape `(N,)` bool.
5. Return `(t_seg, s_line, hit)` where each component has shape `(N,)`. `t_seg = ts[..., 0]`, `s_line = ts[..., 1]`.

Assume every 2×2 is non-singular (the parallel case is covered in the `try-except-solve` atom).

**Do NOT loop over segments.** The whole point is one batched `linalg.solve` call.

Inputs: `S0`, `S1` shape `(N, 2)`; `L0`, `L1` shape `(2,)`.
Output: tuple `(t_seg (N,), s_line (N,), hit (N,) bool)`.

In [ ]:
def ex2_batched_seg_line(S0, S1, L0, L1):
    d = S1 - S0                                       # (N, 2)
    e = L1 - L0                                       # (2,)
    A = t.stack([d, -e.expand_as(d)], dim=-1)         # (N, 2, 2): cols [d, -e]
    b = L0 - S0                                       # (N, 2)
    ts = t.linalg.solve(A, b)                         # (N, 2)
    t_seg = ts[..., 0]                                # (N,)
    s_line = ts[..., 1]                               # (N,)
    hit = (t_seg >= 0.0) & (t_seg <= 1.0)             # (N,) bool
    return t_seg, s_line, hit


<details><summary>Solution</summary>

```python
def ex2_batched_seg_line(S0, S1, L0, L1):
    d = S1 - S0                                       # (N, 2)
    e = L1 - L0                                       # (2,)
    A = t.stack([d, -e.expand_as(d)], dim=-1)         # (N, 2, 2): cols [d, -e]
    b = L0 - S0                                       # (N, 2)
    ts = t.linalg.solve(A, b)                         # (N, 2)
    t_seg = ts[..., 0]                                # (N,)
    s_line = ts[..., 1]                               # (N,)
    hit = (t_seg >= 0.0) & (t_seg <= 1.0)             # (N,) bool
    return t_seg, s_line, hit
```

**Why `t.stack([..., -e.expand_as(d)], dim=-1)`.** The matrix `A` has columns `d` and `-e`. With `d: (N, 2)` and `e: (2,)`, we need to broadcast `-e` along the batch axis before stacking — `.expand_as(d)` does that without allocating new memory. `dim=-1` puts the stacked tensors as COLUMNS (the last axis of the resulting `(N, 2, 2)`). Stacking on `dim=0` would give `(2, N, 2)` — wrong shape for `linalg.solve`.

**Why one batched call beats a Python loop.** PyTorch's `linalg.solve` uses LAPACK's batched routines internally; the overhead per system in a batch of 1000 is roughly the same as one system. A Python loop pays per-iteration Python + kernel-launch overhead for every segment. For 1000 segments the batched form is typically 100-1000x faster.

**Closed interval `[0, 1]`.** Endpoints on the line count as hits. The half-open `[0, 1)` convention appears in raycasting (so consecutive segments don't both claim the shared endpoint) — different drill. For pure intersection testing, closed is the canonical choice.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()